## Part 6 — Q9: text/image causal coupling (no-code runner)

**For Q9 only, start here. Skip all earlier cells, including the older setup and Drive mount.**
Select a GPU runtime (A100 recommended), then run **1. Setup**, choose the model and mode in **2. Choose experiment**, and run **4. Run**, followed by **5. Results**. Step 3 is optional; all fields have preset defaults. Do not use whole-notebook Run all in the main notebook: that would also launch the older experiments.

- **smoke**: first-run wiring/audit check, not scientific evidence.
- **discovery**: clean T5/DiT measurements and calibration; no edited generations.
- **screen**: paired single-forward interventions from clean latent states, not final edited images.
- **confirm**: full paired generations/rescues; the default is **576 edited trajectories**. Enable the explicit confirmation checkbox in step 4 only after reviewing the budget.

No Python edits or external config file are required. Checkpoint, scheduler steps, guidance, prompts, seeds and intervention settings come from the model/mode presets. Optional form fields let you narrow the experiment after screening. Presets are hypotheses, not automatic discovery of the correct sites.

FLUX.1-dev and Schnell are supported. PixArt has no evolving DiT text read-back pathway.
For gated FLUX.1-dev, accept its Hugging Face license and add a Colab secret named **HF_TOKEN** (or use the login widget).
Raw activations/replay states remain in memory, full images stay in temporary `/content`, and Drive export is **off by default**, limited to 250 MiB of compact artifacts per run. Runtime reset loses local work.

[Experimental design and interpretation limits](https://github.com/BrendanGho/massive-activations-fig3/blob/main/SPEC_Q9.md)


In [ ]:
# @title 1. Q9 setup and authentication (self-contained; run once)
import os, subprocess, sys
from pathlib import Path
Q9_REPO_DIR = '/content/massive-activations-fig3'
if not Path(Q9_REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--branch', 'main',
                    'https://github.com/BrendanGho/massive-activations-fig3.git', Q9_REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', Q9_REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(Q9_REPO_DIR)
if Q9_REPO_DIR not in sys.path:
    sys.path.insert(0, Q9_REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[q9]'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, then rerun Q9 setup.')
print(torch.cuda.get_device_name(), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if torch.cuda.memory_allocated() > 2**30:
    print('Other experiments retain GPU memory. For best results, restart the session and run only Part 6.')
from huggingface_hub import login
from google.colab import userdata
try:
    q9_hf_token = userdata.get('HF_TOKEN')
except Exception:
    q9_hf_token = None
if q9_hf_token:
    login(token=q9_hf_token, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)
print('Q9 setup ready. No Drive mount is needed.')


In [ ]:
# @title 2. Choose experiment — these are the only routine settings
Q9_MODEL = 'flux1-dev' # @param ['flux1-dev', 'flux-schnell']
Q9_MODE = 'smoke' # @param ['smoke', 'discovery', 'screen', 'confirm']
Q9_RESOLUTION = 'preset' # @param ['preset', '512', '1024']
# Preset = 1024. A lower resolution changes the experimental setting, not just speed.
q9_advanced = {}
q9_finished = False
q9_result = None
print('Selected:', Q9_MODEL, Q9_MODE, 'resolution:', Q9_RESOLUTION)
print('Preset prompts, seeds, sites and methods will be used. Skip step 3 unless you want overrides.')
print('Rerunning this cell resets any earlier advanced overrides.')


### Optional field guide

Use commas for indices/seeds/methods (for example `17`, `0,14,27`, or `zero,ordinary_zero`).
Use `||` between custom prompts. Calibration and evaluation prompts must not overlap.
Prompt count `0` keeps all preset/custom prompts; a positive number takes the first N. Reducing counts makes a smaller pilot, not full confirmation.

Text methods: `remove_direction,norm_matched,suppress_channel,zero,ordinary_zero,random_direction,donor_swap`.
Routing: `image_reads_text_score,image_reads_text_value,text_reads_register_score,text_reads_register_value,text_reads_content_score,text_reads_content_value`.
Reverse: `image_remove_direction,image_zero,image_ordinary_zero`.
Rescues require confirm mode, text-state methods and a rescue layer downstream of every site. Select rescues `none` for routing/reverse confirmation. Readout must follow every site.
LPIPS/CLIP apply to confirm images; structured scores require an external evaluator CSV and are optional.
If you do not need these controls, **skip both the guide and step 3**.


In [ ]:
# @title 3. Optional advanced form — leave blank/0/preset to keep defaults
# Sites are zero-based blocks (-1 = projected T5); steps are zero-based denoising indices.
Q9_SITES = '' # @param {type:"string"}
Q9_STEPS = '' # @param {type:"string"}
Q9_METHODS = '' # @param {type:"string"}
Q9_PROMPT_COUNT = 0 # @param {type:"integer"}
Q9_SEEDS = '' # @param {type:"string"}
Q9_CUSTOM_PROMPTS = '' # @param {type:"string"}
Q9_CALIBRATION_PROMPT_COUNT = 0 # @param {type:"integer"}
Q9_CALIBRATION_SEEDS = '' # @param {type:"string"}
Q9_CUSTOM_CALIBRATION_PROMPTS = '' # @param {type:"string"}
Q9_CANDIDATE_CLASSES = 'eos,pad' # @param ['eos,pad', 'content', 'content,eos,pad,special']
Q9_CANDIDATE_SOURCE = 'union' # @param ['norm', 'sink', 'union', 'intersection']
Q9_RESCUES = 'preset' # @param ['preset', 'none', 'none,projection,state,ordinary_projection,sham']
Q9_RESCUE_LAYER = '' # @param {type:"string"}
Q9_READOUT_LAYER = '' # @param {type:"string"}
Q9_ATTENTION_LAYERS = '' # @param {type:"string"}
Q9_INCLUDE_EMPTY = True # @param {type:"boolean"}
Q9_MEMORY = 'auto' # @param ['auto', 'offload', 'gpu']
Q9_OPTIMIZE_PROBES = True # @param {type:"boolean"}
Q9_SKIP_UNAVAILABLE = True # @param {type:"boolean"}
Q9_EVALUATE_LPIPS = True # @param {type:"boolean"}
Q9_EVALUATE_CLIP = True # @param {type:"boolean"}
Q9_STRUCTURED_SCORES = '' # @param {type:"string"}
q9_advanced = dict(
    sites=Q9_SITES, steps=Q9_STEPS, methods=Q9_METHODS, prompt_count=Q9_PROMPT_COUNT,
    seeds=Q9_SEEDS, prompts=Q9_CUSTOM_PROMPTS, calibration_prompt_count=Q9_CALIBRATION_PROMPT_COUNT,
    calibration_seeds=Q9_CALIBRATION_SEEDS, calibration_prompts=Q9_CUSTOM_CALIBRATION_PROMPTS,
    candidate_classes=Q9_CANDIDATE_CLASSES, candidate_source=Q9_CANDIDATE_SOURCE,
    rescues=Q9_RESCUES, rescue_layer=Q9_RESCUE_LAYER, readout_layer=Q9_READOUT_LAYER,
    attention_layers=Q9_ATTENTION_LAYERS, include_empty=Q9_INCLUDE_EMPTY, memory=Q9_MEMORY,
    optimize_probes=Q9_OPTIMIZE_PROBES, skip_unavailable=Q9_SKIP_UNAVAILABLE,
    evaluate_lpips=Q9_EVALUATE_LPIPS, evaluate_clip=Q9_EVALUATE_CLIP,
    structured_scores=Q9_STRUCTURED_SCORES)
print('Advanced overrides recorded. Step 4 validates them before loading a model.')


In [ ]:
# @title 4. Validate, show budget and run Q9
Q9_RUN_EXPERIMENT = True # @param {type:"boolean"}
Q9_CONFIRM_FULL_RUN = False # @param {type:"boolean"}
# Confirm may take a long time. Enable its checkbox only after reviewing the printed budget.
import json
from dataclasses import asdict
from src.experiments.q9_colab import build_config, run_budget
if 'Q9_MODEL' not in globals():
    raise RuntimeError('Run Q9 steps 1 and 2 first.')
q9_finished = False
cfg = build_config(Q9_MODEL, Q9_MODE, Q9_RESOLUTION,
                   vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,
                   bf16=torch.cuda.is_bf16_supported(), advanced=q9_advanced)
print(json.dumps(run_budget(cfg), indent=2))
print('Upper bounds before resume/unavailable skips; donor replays may add forwards.')
print('Sites:', cfg.sites, '| steps:', cfg.steps, '| methods:', cfg.methods, '| rescues:', cfg.rescues)
print('Evaluation prompts:', cfg.prompts, '| seeds:', cfg.seeds)
print('Dtype:', cfg.dtype, '| offload:', cfg.offload, '| local outputs:', cfg.output_dir)
Q9_CONFIG_PATH = f'/content/q9_{cfg.model_preset}_{cfg.mode}_config.json'
Path(Q9_CONFIG_PATH).write_text(json.dumps(asdict(cfg), indent=2), encoding='utf-8')
if not Q9_RUN_EXPERIMENT:
    print('Preview only. Enable Q9_RUN_EXPERIMENT and rerun this cell to execute.')
elif cfg.mode == 'confirm' and not Q9_CONFIRM_FULL_RUN:
    print('Not started. Review the budget, enable Q9_CONFIRM_FULL_RUN, then rerun this cell.')
else:
    subprocess.run([sys.executable, '-u', '-m', 'src.experiments.text_image_coupling',
                    '--config', Q9_CONFIG_PATH], check=True)
    q9_finished = True
    print('Finished. Run step 5 to view figures.')


In [ ]:
# @title 5. Show Q9 results and figures
from IPython.display import display, Image
if not globals().get('q9_finished', False):
    print('No completed run in this workflow yet. Run step 4 first.')
else:
    from src.experiments.q9_report import locate
    q9_result = locate(cfg)
    print((q9_result / 'report_status.json').read_text())
    for figure in sorted((q9_result / 'figures').glob('*.png')):
        print(figure.name)
        display(Image(filename=str(figure)))
    print('Results:', q9_result)
    print('Check audits.csv and direction_stability.csv before interpreting a negative result.')
    print('No-candidate/unavailable controls are excluded. Smoke is a wiring check only.')


In [ ]:
# @title 6. Optional compact Drive export (no raw activations or full image grid)
Q9_EXPORT_TO_DRIVE = False # @param {type:"boolean"}
Q9_DRIVE_ROOT = '/content/drive/MyDrive/Research/MA/q9_compact' # @param {type:"string"}
if not Q9_EXPORT_TO_DRIVE:
    print('Drive export off. Results remain in temporary Colab storage.')
elif not globals().get('q9_finished', False):
    print('Run step 4 successfully before exporting.')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'src.experiments.text_image_coupling',
                    '--config', Q9_CONFIG_PATH, '--export-compact', Q9_DRIVE_ROOT], check=True)
